# Predição do Valor de Imóveis na Cidade de São Paulo

**Projeto de Regressão**

## Objetivo

Prever dois valores de imóveis em São Paulo:

1. Preço de **venda**;
2. Preço de **aluguel**.

Usamos a métrica **MAE (Erro Absoluto Médio)** para avaliar os modelos: quanto menor, melhor.

## Dataset

- Fonte: https://www.kaggle.com/datasets/argonalyst/sao-paulo-real-estate-sale-rent-april-2019
- Arquivo: `sao-paulo-properties-april-2019.csv` (baixe manualmente e coloque na mesma pasta
  deste notebook, pois o Kaggle exige login).
- Colunas usadas: `Price`, `Condo`, `Size`, `Rooms`, `Toilets`, `Suites`, `Parking`,
  `Elevator`, `Furnished`, `Swimming Pool`, `New`, `Latitude`, `Longitude`,
  `Negotiation Type` (`sale` ou `rent`).

## Bibliotecas

Somente `pandas`, `numpy` e `matplotlib`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## 1. Carregando os dados

In [ ]:
dados = pd.read_csv("sao-paulo-properties-april-2019.csv")
dados.head()


In [ ]:
dados.info()


In [ ]:
dados.describe()


In [ ]:
# Verificando valores nulos
dados.isnull().sum()


In [ ]:
# Separando venda e aluguel
dados_venda = dados[dados["Negotiation Type"] == "sale"].copy()
dados_aluguel = dados[dados["Negotiation Type"] == "rent"].copy()

print("Imóveis de venda:", len(dados_venda))
print("Imóveis de aluguel:", len(dados_aluguel))


## 2. Visualização dos dados

In [ ]:
# Histograma do preço de venda
plt.hist(dados_venda["Price"], bins=50)
plt.title("Distribuição do Preço de Venda")
plt.xlabel("Preço (R$)")
plt.ylabel("Quantidade de imóveis")
plt.show()


In [ ]:
# Histograma do preço de aluguel
plt.hist(dados_aluguel["Price"], bins=50, color="orange")
plt.title("Distribuição do Preço de Aluguel")
plt.xlabel("Preço (R$)")
plt.ylabel("Quantidade de imóveis")
plt.show()


In [ ]:
# Tamanho x Preço (venda)
plt.scatter(dados_venda["Size"], dados_venda["Price"], alpha=0.3, s=10)
plt.title("Tamanho x Preço de Venda")
plt.xlabel("Tamanho (m²)")
plt.ylabel("Preço (R$)")
plt.show()


In [ ]:
# Preço médio de venda por distrito (10 mais caros)
preco_por_distrito = dados_venda.groupby("District")["Price"].mean().sort_values(ascending=False)

plt.barh(preco_por_distrito.head(10).index, preco_por_distrito.head(10).values)
plt.title("10 distritos com maior preço médio de venda")
plt.xlabel("Preço médio (R$)")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Mapa: localização dos imóveis coloridos pelo preço (padrão regional)
plt.figure(figsize=(8, 8))
grafico = plt.scatter(
    dados_venda["Longitude"],
    dados_venda["Latitude"],
    c=dados_venda["Price"],
    cmap="viridis",
    alpha=0.5,
    s=10
)
plt.colorbar(grafico, label="Preço (R$)")
plt.title("Distribuição geográfica do preço de venda em São Paulo")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


In [ ]:
# Correlação entre as variáveis numéricas (dados de venda)
colunas_numericas = ["Price", "Condo", "Size", "Rooms", "Toilets",
                      "Suites", "Parking", "Elevator", "Swimming Pool"]

correlacao = dados_venda[colunas_numericas].corr()

plt.figure(figsize=(7, 6))
plt.imshow(correlacao, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlação")
plt.xticks(range(len(colunas_numericas)), colunas_numericas, rotation=90)
plt.yticks(range(len(colunas_numericas)), colunas_numericas)
plt.title("Correlação entre variáveis")
plt.show()

correlacao


## 3. Preparando os dados para o modelo

Vamos usar apenas as colunas numéricas (sem transformar o distrito em texto/números extras,
para manter o código simples). A latitude e a longitude já ajudam a representar a localização.


In [ ]:
colunas_features = ["Condo", "Size", "Rooms", "Toilets", "Suites", "Parking",
                     "Elevator", "Furnished", "Swimming Pool", "New",
                     "Latitude", "Longitude"]

def preparar_dados(df):
    X = df[colunas_features].to_numpy(dtype=float)
    y = df["Price"].to_numpy(dtype=float)
    return X, y

X_venda, y_venda = preparar_dados(dados_venda)
X_aluguel, y_aluguel = preparar_dados(dados_aluguel)

print("X_venda:", X_venda.shape)
print("X_aluguel:", X_aluguel.shape)


In [ ]:
# Função simples para dividir em treino (80%) e teste (20%)
def dividir_treino_teste(X, y, proporcao_teste=0.2, semente=42):
    np.random.seed(semente)
    indices = np.random.permutation(len(X))
    tamanho_teste = int(len(X) * proporcao_teste)

    indices_teste = indices[:tamanho_teste]
    indices_treino = indices[tamanho_teste:]

    return X[indices_treino], X[indices_teste], y[indices_treino], y[indices_teste]

X_treino_venda, X_teste_venda, y_treino_venda, y_teste_venda = dividir_treino_teste(X_venda, y_venda)
X_treino_aluguel, X_teste_aluguel, y_treino_aluguel, y_teste_aluguel = dividir_treino_teste(X_aluguel, y_aluguel)

print("Treino venda:", len(X_treino_venda), " Teste venda:", len(X_teste_venda))
print("Treino aluguel:", len(X_treino_aluguel), " Teste aluguel:", len(X_teste_aluguel))


## 4. Modelo de Regressão Linear (implementado com numpy)

Vamos treinar uma Regressão Linear "na mão", usando a equação normal:

$$ w = (X^T X)^{-1} X^T y $$

Para isso, adicionamos uma coluna de 1s em X (representa o intercepto/bias do modelo).


In [ ]:
# Adiciona uma coluna de 1s (intercepto) e calcula os pesos com numpy
def treinar_regressao_linear(X, y):
    X_com_bias = np.column_stack([np.ones(len(X)), X])
    pesos = np.linalg.lstsq(X_com_bias, y, rcond=None)[0]
    return pesos

def prever(X, pesos):
    X_com_bias = np.column_stack([np.ones(len(X)), X])
    return X_com_bias @ pesos

def calcular_mae(y_real, y_previsto):
    return np.mean(np.abs(y_real - y_previsto))


In [ ]:
# Treinando o modelo de VENDA
pesos_venda = treinar_regressao_linear(X_treino_venda, y_treino_venda)
previsoes_venda = prever(X_teste_venda, pesos_venda)
mae_venda = calcular_mae(y_teste_venda, previsoes_venda)

print(f"MAE (Venda): R$ {mae_venda:,.2f}")


In [ ]:
# Treinando o modelo de ALUGUEL
pesos_aluguel = treinar_regressao_linear(X_treino_aluguel, y_treino_aluguel)
previsoes_aluguel = prever(X_teste_aluguel, pesos_aluguel)
mae_aluguel = calcular_mae(y_teste_aluguel, previsoes_aluguel)

print(f"MAE (Aluguel): R$ {mae_aluguel:,.2f}")


In [ ]:
# Preço real x preço previsto (venda)
plt.scatter(y_teste_venda, previsoes_venda, alpha=0.3, s=10)
plt.plot([y_teste_venda.min(), y_teste_venda.max()],
         [y_teste_venda.min(), y_teste_venda.max()],
         color="red", linestyle="--")
plt.title("Preço real x Preço previsto (Venda)")
plt.xlabel("Preço real (R$)")
plt.ylabel("Preço previsto (R$)")
plt.show()


## 5. Explicabilidade

Como o modelo é uma Regressão Linear, cada variável tem um **peso (coeficiente)**. O sinal
mostra se a variável aumenta (+) ou diminui (-) o preço, e o tamanho do peso mostra o quanto
ela pesa na decisão do modelo.


In [ ]:
# Pesos do modelo de venda (sem contar o intercepto, que é o primeiro valor)
pesos_venda_series = pd.Series(pesos_venda[1:], index=colunas_features).sort_values()

plt.barh(pesos_venda_series.index, pesos_venda_series.values)
plt.title("Peso de cada variável no preço de VENDA")
plt.xlabel("Peso (coeficiente)")
plt.show()

pesos_venda_series


In [ ]:
# Pesos do modelo de aluguel
pesos_aluguel_series = pd.Series(pesos_aluguel[1:], index=colunas_features).sort_values()

plt.barh(pesos_aluguel_series.index, pesos_aluguel_series.values)
plt.title("Peso de cada variável no preço de ALUGUEL")
plt.xlabel("Peso (coeficiente)")
plt.show()

pesos_aluguel_series


## 6. Conclusão

- Exploramos os dados de imóveis de São Paulo e visualizamos padrões de preço por região e
  por distrito;
- Treinamos um modelo de Regressão Linear (implementado com numpy) para prever o preço de
  venda e o preço de aluguel;
- Avaliamos os modelos com o MAE;
- Analisamos os pesos do modelo para entender quais variáveis mais influenciam o preço.
